# my_open_space — 영상 → 3D Gaussian Splatting 파이프라인

무료 T4 GPU에서 실행합니다. 위쪽 메뉴 **런타임 > 런타임 유형 변경 > T4 GPU** 확인 후 순서대로 셀을 실행하세요.

흐름: 영상 업로드 → 프레임 24장 추출 → COLMAP(카메라 위치 추정) → 카메라 퍼진 정도 확인(0.1 기준) → 3DGS 학습 → `scene.ply` 다운로드.

`scene.ply`를 받은 뒤에는 로컬 저장소의 `compare-formats.mjs` 로 `.compressed.ply` / `.sog` / `.spz` 변환과 비교표 작성을 진행합니다.

## 0. GPU 확인

In [ ]:
!nvidia-smi

## 1. 영상 업로드
아래 셀을 실행하면 파일 선택 창이 뜹니다. 야외 공간을 찍은 영상 파일(15~20초, 1080p)을 올리세요.

In [ ]:
from google.colab import files
uploaded = files.upload()
VIDEO_PATH = list(uploaded.keys())[0]
print('업로드된 파일:', VIDEO_PATH)

## 2. 프레임 24장 추출
영상 길이를 재서 24장이 고르게 뽑히도록 fps를 역산합니다. 1080p로 맞추고 여백은 검게 채웁니다.

In [ ]:
import subprocess, os

PROJECT = '/content/scene'
os.makedirs(f'{PROJECT}/input', exist_ok=True)
N_FRAMES = 24

probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'csv=p=0', VIDEO_PATH],
    capture_output=True, text=True
)
duration = float(probe.stdout.strip())
fps = N_FRAMES / duration
print(f'영상 길이 {duration:.1f}초 -> {N_FRAMES}장을 뽑기 위한 fps={fps:.4f}')

!ffmpeg -y -i "{VIDEO_PATH}" -vf "fps={fps},scale=1920:1080:force_original_aspect_ratio=decrease,pad=1920:1080:(ow-iw)/2:(oh-ih)/2" -qscale:v 2 {PROJECT}/input/frame_%03d.jpg

n = len([f for f in os.listdir(f'{PROJECT}/input') if f.endswith('.jpg')])
print(f'추출된 프레임 수: {n} (목표 24)')
assert 20 <= n <= 28, '프레임 수가 24장에서 크게 벗어났습니다 — 영상 길이/코덱을 확인하세요.'

## 3. 3D Gaussian Splatting 저장소 준비
COLMAP 실행 스크립트(`convert.py`)와 학습 스크립트(`train.py`)가 이 저장소에 들어 있습니다. CUDA 확장 컴파일이 T4에서 몇 분 걸립니다.

In [ ]:
%cd /content
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting
%cd /content/gaussian-splatting
!apt-get -qq update && apt-get -qq install -y colmap
!pip install -q plyfile tqdm

# 최신 Colab 이미지(gcc/nvcc 조합)에서는 diff-gaussian-rasterization의 헤더가
# <cstdint>를 안 받아와서 uint32_t/uint64_t/std::uintptr_t 컴파일 에러가 난다.
# 빌드 전에 include를 하나 추가해서 미리 막는다.
!sed -i '1i #include <cstdint>' submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h
!pip install -q ./submodules/diff-gaussian-rasterization ./submodules/simple-knn

> 그래도 CUDA 확장 설치에서 에러가 나면: 에러 메시지에서 실제 컴파일 에러 줄(`error:`가 포함된 줄)을 찾아 알려주세요. torch/CUDA 버전 불일치라면 **런타임 > 런타임 다시 시작** 후 이 셀부터 재실행하는 것도 방법입니다.

## 4. COLMAP으로 카메라 위치 추정
특징점 추출 → 매칭 → 카메라 포즈 계산(sparse reconstruction) → 왜곡 보정까지 한 번에 수행합니다.

In [ ]:
%cd /content/gaussian-splatting
# COLMAP이 내부적으로 쓰는 Qt가 디스플레이 없는 Colab에서 GUI 플랫폼을 초기화하려다 죽는 것을 막는다.
# --no_gpu: SiftGPU가 필요로 하는 OpenGL 컨텍스트도 headless에선 못 만드므로 CPU 특징점 추출로 전환.
# 24장짜리 작은 장면이라 CPU로도 충분히 빠르다.
!QT_QPA_PLATFORM=offscreen python convert.py -s {PROJECT} --no_gpu

## 5. 카메라가 퍼진 정도 확인 (기준선 0.1)
카메라 중심 위치들이 퍼진 정도를, sparse 점군의 장면 크기로 정규화해서 잽니다. 0.1 미만이면 시차가 부족한 것이니 몇 걸음 더 옮겨 다니며 다시 찍는 편이 낫습니다.

In [ ]:
import sys, numpy as np
sys.path.append('/content/gaussian-splatting')
from scene.colmap_loader import read_extrinsics_binary, read_points3D_binary

images = read_extrinsics_binary(f'{PROJECT}/sparse/0/images.bin')
centers = []
for key in images:
    im = images[key]
    R = im.qvec2rotmat()
    t = np.array(im.tvec)
    centers.append(-R.T @ t)
centers = np.array(centers)

points, _, _ = read_points3D_binary(f'{PROJECT}/sparse/0/points3D.bin')
scene_extent = float(np.linalg.norm(points.max(axis=0) - points.min(axis=0)))
cam_spread = float(np.linalg.norm(centers.std(axis=0))) / scene_extent

print(f'등록된 카메라 수: {len(centers)}')
print(f'장면 크기(대각선): {scene_extent:.3f}')
print(f'카메라가 퍼진 정도(정규화): {cam_spread:.4f}  (기준선 0.1)')
if cam_spread < 0.1:
    print('경고: 0.1 미만입니다. 이 경로로는 시차가 부족할 수 있습니다 — 재촬영을 권장합니다.')
else:
    print('통과: 카메라가 충분히 퍼져 있습니다.')

## 6. 3D Gaussian Splatting 학습
T4에서 24장짜리 작은 장면 기준 7000 iteration이면 수 분~십수 분 내로 끝납니다.

In [ ]:
%cd /content/gaussian-splatting
!python train.py -s {PROJECT} -m {PROJECT}/output --iterations 7000 --save_iterations 7000 --test_iterations 7000

## 7. 결과 ply 확인 및 다운로드
학습된 알갱이(gaussian) 개수를 세고, `scene.ply` 로 복사해서 다운로드합니다. 이 파일을 로컬 `my_open_space` 폴더에 넣고 `node compare-formats.mjs scene.ply` 를 실행하면 형식별 비교표가 나옵니다.

In [ ]:
import glob, shutil
from plyfile import PlyData

ply_path = sorted(glob.glob(f'{PROJECT}/output/point_cloud/iteration_*/point_cloud.ply'))[-1]
ply = PlyData.read(ply_path)
n_gaussians = ply['vertex'].count
print(f'학습된 알갱이 개수: {n_gaussians:,}')

shutil.copy(ply_path, '/content/scene.ply')
from google.colab import files
files.download('/content/scene.ply')

## 7-1. scene.sog 변환 (T4 GPU에서)
로컬 PC의 내장 그래픽카드가 SOG 압축(색상 k-means)을 못 버티고 죽는 경우가 있습니다. Colab의 T4는 진짜 서버용 GPU라 이 단계가 문제없이 돌아가니, 여기서 바로 `.sog` 까지 만들어서 다운로드합니다.

In [ ]:
# Node.js 설치 (Colab 기본 이미지엔 없음)
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get -qq install -y nodejs > /dev/null 2>&1
!node -v

# T4 GPU로 scene.ply -> scene.sog 변환
!npx -y @playcanvas/splat-transform@latest -w /content/scene.ply /content/scene.sog

from google.colab import files
files.download('/content/scene.sog')

## 다음 단계 (로컬에서)
1. 다운로드된 `scene.sog`를 `my_open_space/scene.sog`로 저장 (7-1번 셀에서 이미 만들어짐 — 로컬 GPU가 필요 없음)
2. `scene.ply`는 이미 있다면 그대로 두고, 없다면 7번 셀 결과도 받아서 `my_open_space/scene.ply`로 저장
3. `node compare-formats.mjs scene.ply` 실행 → `.compressed.ply`/`.spz` 변환 + 비교표 (`.sog`는 이미 있으니 재변환하지 않음)
4. README에 표와 수치 채우기, `scene.sog` 포함해서 커밋/푸시
5. GitHub Pages 배포 및 `/scene.sog` 직접 다운로드 확인